# Spatial Intelligence - Part 2

In [2]:
# This cell is not needed if you have pip installed topologicpy
#import sys
#sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

## 1. Import the needed libraries

In [3]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

c:\Users\Sushmitha\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Check the TopologicPy Version

In [4]:
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

This tutorial requires topologicpy version 0.9.18 or newer.
The version that you are using (0.9.29) is EQUAL TO the latest version available on PyPI.


## 3. Set your renderer:
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [5]:
renderer = "vscode"

## 4. Utility functions to reset the face dictionaries and transfer dictionaries by key

In [6]:
def reset_dictionaries(shell):
    faces = Topology.Faces(shell)
    for i, f in enumerate(faces):
        d = Topology.Dictionary(f)
        keys = Dictionary.Keys(d)
        for key in keys:
            if not key == "face_id":
                d = Dictionary.RemoveKey(d, key)
        f = Topology.SetDictionary(f, d)

def transfer_dicts_by_key(topologies, selectors, key):
    dicts = {}
    for t in topologies:
        d = Topology.Dictionary(t)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            dicts[str(value)] = t
    
    for s in selectors:
        d = Topology.Dictionary(s)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            f = dicts.get(str(value), None)
            if f:
                f = Topology.SetDictionary(f, d)


## 5. Import the gallery floor plan

In [7]:
floor_plan = Topology.ByBREPPath(r"D:\3rd sem\GRAPH MACHINE LEARNING\ASSIGNMENTS\ASSIGNMENT-2\notebook-2\Graph-machine-learning\2.Resources\Market.brep")
triangles = Cluster.Faces(floor_plan)
shell = Shell.ByFaces(triangles)
eb = Shell.ExternalBoundary(shell)
ib_list = Shell.InternalBoundaries(shell)
new_face = Face.ByWires(eb, ib_list)
market = Topology.RemoveCollinearEdges(new_face)
print(market)



## 6. Show the geometry

In [8]:
Topology.Show(market,
              camera=[0,0,7],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor="white",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer = renderer)

## 7. Create a grid overlay

In [9]:
faces = Topology.Faces(market)

print("Number of faces:", len(faces))

Topology.Faces - Warning: The input is a Face. Returning the same face embedded in a list.
caller name: <module>
Number of faces: 1


In [10]:
import numpy as np
b_r = Wire.BoundingRectangle(market)
d = Topology.Dictionary(b_r)
xmin = Dictionary.ValueAtKey(d, "xmin")
xmax = Dictionary.ValueAtKey(d, "xmax")
ymin = Dictionary.ValueAtKey(d, "ymin")
ymax = Dictionary.ValueAtKey(d, "ymax")
width = Dictionary.ValueAtKey(d, "width")
length = Dictionary.ValueAtKey(d, "length")
uRange = list(np.arange(0, width + 6, 6))
vRange = list(np.arange(0, length + 6, 6))

grid = Grid.EdgesByDistances(market, clip=True, uRange=uRange, vRange=vRange)

## 8. Show the geometry and the grid

In [11]:
Topology.Show(market, grid,
              camera=[0,0,7],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor="grey",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer = renderer)

## 9. Slice the floor plan with the grid to create a topologic shell

In [12]:
shell = Topology.Slice(market, grid)
faces = Topology.Faces(shell)
# Assign a sequential unique face id to reference it later (e.g. "face_21")
for i, f in enumerate(faces):
    d = Dictionary.ByKeyValue("face_id", "face_"+str(i+1))
    f = Topology.SetDictionary(f, d)

## 10. Derive analysis graphs from the shell

In [13]:
# Note: Graph nodes automatically inherit the dictionaries of the entities they 
navigation_graph = Graph.ByTopology(shell, direct=False, viaSharedTopologies=True)
analysis_graph = Graph.ByTopology(shell)

In [14]:
Topology.Show(shell,
              camera=[0,0,7],
              faceColor=[210,210,250],
              faceOpacity=0.9,
              edgeColor="black",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer = renderer)

## 11. Derive and store the graph vertices

In [15]:
# Note: Graph nodes automatically inherit the dictionaries of the entities they 

analysis_graph = Graph.ByTopology(shell)

In [16]:
g_verts = Graph.Vertices(analysis_graph)

In [17]:
Topology.Show(analysis_graph, 
              camera=[0,0,7],
              vertexSize=4,
              vertexColor="red",
              edgeColor="lightgrey",
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

## 12. Spatial Intelligence through Graph Analysis

### a. Community Detection (About 5 minutes)

In [18]:
community_list = Graph.CommunityPartition(analysis_graph, colorScale="thermal")

In [19]:
reset_dictionaries(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

In [20]:
Topology.Show(faces,
              faceColorKey="cp_color",
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

### b. Degree centrality

Bin By Dictionary Key
* Use the community partition number (or colour) to separate the faces of the shell into different bins or categories
* Derive the outer boundary of each face group (perimeter) and make a face out of that perimeter

In [21]:
bins = Topology.BinByDictionaryKey(faces, key="community")
bin_dict = bins[0]
keys = list(bin_dict.keys())
face_groups = []
for key in keys:
    bin_faces = bin_dict[key]
    temp_shell = Shell.ByFaces(bin_faces)
    eb = Shell.ExternalBoundary(temp_shell)
    eb = Wire.RemoveCollinearEdges(eb)
    eb = Face.ByWire(eb)
    face_groups.append(eb)



Show the result

In [22]:
Topology.Show(face_groups,
              faceOpacity=1,
              showEdges=True,
              edgeWidth=8,
              edgeColor="grey",
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

Create a new shell from the new faces

In [23]:
new_shell = Shell.ByFaces(face_groups)

Show the result

In [24]:
Topology.Show(new_shell,
              faceOpacity=0.9,
              showEdges=True,
              showVertices=True,
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

Create a new graph from the new shell

In [25]:
new_graph = Graph.ByTopology(new_shell)
new_verts = Graph.Vertices(new_graph)
for v in new_verts:
    d = Dictionary.ByKeysValues(["color", "size"], ["red", 10])
    v = Topology.SetDictionary(v, d)

Show the result

In [26]:
Topology.Show(new_shell, new_graph,
              faceOpacity=0.9,
              showEdges=True,
              showVertices=True,
              vertexSizeKey="size",
              vertexColorKey="color",
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

Compute degree centralities

In [27]:
degree_centralities = Graph.DegreeCentrality(new_graph, normalize=False)

Transfer/Interpolate values from the new graph vertices to the original graph vertices

In [28]:
for v in g_verts:
    new_v = Vertex.InterpolateValue(v, vertices=new_verts, n=3, key="degree_centrality")

Derive the colour of each vertex based on the interpolated value

In [29]:
minValue = min(degree_centralities)
maxValue = max(degree_centralities)
for v in g_verts:
    d = Topology.Dictionary(v)
    d_c = Dictionary.ValueAtKey(d, "degree_centrality")
    color = Color.AnyToHex(Color.ByValueInRange(d_c, minValue=minValue, maxValue=maxValue, colorScale="thermal"))
    d = Dictionary.SetValueAtKey(d, "dc_color", color)
    d = Dictionary.SetValueAtKey(d, "size", 16)
    v = Topology.SetDictionary(v, d)

Transfer the information from the graph vertices to the faces of the original shell

In [30]:
reset_dictionaries(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

Show the result

In [31]:
Topology.Show(faces,
              faceColorKey="dc_color",
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              vertexSizeKey="size",
              vertexColorKey="dc_color",
              camera=[0,0,7],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

## 13. CALCULATIONS OF DEGREE CENTRALITY

In [32]:
from statistics import mean

dc_values = []

for v in new_verts:

    d = Topology.Dictionary(v)

    dc = Dictionary.ValueAtKey(
        d,
        "degree_centrality"
    )

    if dc is not None:
        dc_values.append(dc)

dc_min = min(dc_values)
dc_max = max(dc_values)
dc_mean = mean(dc_values)

print("DEGREE CENTRALITY")
print("Min :", round(dc_min,4))
print("Max :", round(dc_max,4))
print("Mean:", round(dc_mean,4))
print("Spaces analysed:", len(dc_values))

DEGREE CENTRALITY
Min : 0.0625
Max : 0.375
Mean: 0.1838
Spaces analysed: 17


In [34]:
vertex_data = []

for v in new_verts:

    d = Topology.Dictionary(v)

    dc = Dictionary.ValueAtKey(
        d,
        "degree_centrality"
    )

    if dc is not None:

        x = round(Vertex.X(v),2)
        y = round(Vertex.Y(v),2)

        vertex_data.append([x,y,dc])

vertex_data = sorted(
    vertex_data,
    key=lambda x: x[2],
    reverse=True
)

print("\nTOP 5 MOST CONNECTED SPACES\n")

for i, item in enumerate(vertex_data[:5]):

    print(
        f"{i+1}. Vertex at ({item[0]}, {item[1]}) --> DC: {round(item[2],4)}"
    )


TOP 5 MOST CONNECTED SPACES

1. Vertex at (29.79, 18.76) --> DC: 0.375
2. Vertex at (1.87, 5.8) --> DC: 0.3125
3. Vertex at (5.09, 32.95) --> DC: 0.3125
4. Vertex at (5.99, 57.32) --> DC: 0.3125
5. Vertex at (26.85, -23.27) --> DC: 0.3125


In [35]:
print("\nTOP 5 LEAST CONNECTED SPACES\n")

for i, item in enumerate(vertex_data[-5:]):

    print(
        f"{i+1}. Vertex at ({item[0]}, {item[1]}) --> DC: {round(item[2],4)}"
    )


TOP 5 LEAST CONNECTED SPACES

1. Vertex at (-32.93, 83.12) --> DC: 0.0625
2. Vertex at (49.02, -7.13) --> DC: 0.0625
3. Vertex at (51.09, -42.46) --> DC: 0.0625
4. Vertex at (52.57, 16.26) --> DC: 0.0625
5. Vertex at (48.73, -25.3) --> DC: 0.0625


In [36]:
hub_spaces = [
    item for item in vertex_data
    if item[2] > dc_mean
]

print("\nHUB SPACES")
print("Highly connected spaces:", len(hub_spaces))


HUB SPACES
Highly connected spaces: 9


In [47]:
# ==========================================
# VISUALISE HUB SPACES
# ==========================================

hub_vertices = []

for item in hub_spaces:

    x = item[0]
    y = item[1]

    v = Vertex.ByCoordinates(x, y, 0)

    d = Dictionary.ByKeysValues(
        ["color", "size"],
        ["red", 16]
    )

    v = Topology.SetDictionary(v, d)

    hub_vertices.append(v)

Topology.Show(
    shell,
    hub_vertices,
    faceOpacity=0.2,
    camera=[0,0,7],
    showEdges=True,
    vertexColorKey="color",
    vertexSizeKey="size",
    backgroundColor="black",
    width=900,
    height=700,
    renderer=renderer
)

In [49]:
# ==========================================
# DEAD-END / LOW CONNECTIVITY SPACES
# ==========================================

dead_ends = [
    item for item in vertex_data
    if item[2] < (dc_mean * 0.4)
]

print("\nLOW CONNECTIVITY SPACES")
print("Poorly connected spaces:", len(dead_ends))


LOW CONNECTIVITY SPACES
Poorly connected spaces: 6


In [59]:
# ==========================================
# VISUALISE DEAD-END SPACES
# ==========================================

dead_vertices = []

for item in dead_ends:

    x = item[0]
    y = item[1]

    v = Vertex.ByCoordinates(x, y, 0)

    d = Dictionary.ByKeysValues(
        ["color", "size"],
        ["blue", 14]
    )

    v = Topology.SetDictionary(v, d)

    dead_vertices.append(v)

Topology.Show(
    shell,
    dead_vertices,
    faceOpacity=0.2,
    camera=[0,0,7], 
    showEdges=True,
    vertexColorKey="color",
    vertexSizeKey="size",
    backgroundColor="black",
    width=900,
    height=700,
    renderer=renderer
)